
# 09 — Final publication tables, figures, and manuscript-ready results

This notebook performs **publication synthesis only**. It does not alter any frozen parameter, model, payload, split, or inclusion rule.

The evidence chain is:

1. exact reversible embedding and net-capacity accounting;
2. frozen allocation weight alpha = 0.25;
3. 40,000-case held-out test experiment;
4. independent enhanced-CNN steganalysis;
5. end-to-end computational-cost and side-information analysis.

The source dataset used in this experimental series is the **BOSSbase-derived grayscale JPEG QF95 collection at 256×256 pixels**. Exact reversibility refers to the decoded grayscale pixel array used by the RDH experiment, not to restoration of the original pre-JPEG source file.

The notebook creates publication-oriented CSV/LaTeX tables, PNG/SVG figures, environment metadata, and concise manuscript-ready text. No post-test retuning is permitted.


In [ ]:

from pathlib import Path
import json, platform, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from rdhlab.publication_synthesis import validate_grid, compact_metric_table, primary_text

ROOT=Path('/workspace')
OUT=ROOT/'results'/'publication'
OUT.mkdir(parents=True,exist_ok=True)

p06=ROOT/'results'/'frozen_test_final'
p07=ROOT/'results'/'final_steganalysis'
p08=ROOT/'results'/'robustness_resources'

rdh=pd.read_csv(p06/'summary_common_feasible.csv')
det=pd.read_csv(p07/'enhanced_cnn_test_summary.csv')
paired=pd.read_csv(p07/'paired_joint_vs_baselines.csv')
primary=json.loads((p07/'primary_endpoint.json').read_text())
timing=pd.read_csv(p08/'end_to_end_timing_summary.csv')
side=pd.read_csv(p08/'sideinfo_summary.csv')
resource=json.loads((p08/'resource_run_complete.json').read_text())
allocator=json.loads((ROOT/'config'/'frozen_allocator.json').read_text())
config=yaml.safe_load((ROOT/'config'/'experiment.yaml').read_text())

validate_grid(rdh)
validate_grid(det)
validate_grid(side)

assert primary['decision']=='TEST_DETECTABILITY_PRIMARY_CONFIRMED'
assert primary['test_split_used_for_retuning'] is False
assert primary['no_retuning_permitted'] is True
assert resource['status']=='COMPLETE'
assert resource['no_retuning_permitted'] is True

print(primary_text(primary))
print('Publication output:',OUT)


In [ ]:

# Capture computational environment for reproducibility.
try:
    import torch
    torch_version=torch.__version__
    cuda_available=bool(torch.cuda.is_available())
    cuda_name=torch.cuda.get_device_name(0) if cuda_available else None
except Exception:
    torch_version=None; cuda_available=False; cuda_name=None

try:
    import psutil
    cpu_count_logical=psutil.cpu_count(logical=True)
    cpu_count_physical=psutil.cpu_count(logical=False)
    ram_gib=psutil.virtual_memory().total/(1024**3)
except Exception:
    cpu_count_logical=cpu_count_physical=None
    ram_gib=None

env={
    'platform':platform.platform(),
    'machine':platform.machine(),
    'processor':platform.processor(),
    'python':sys.version,
    'torch':torch_version,
    'cuda_available':cuda_available,
    'cuda_device':cuda_name,
    'cpu_logical':cpu_count_logical,
    'cpu_physical':cpu_count_physical,
    'ram_gib':ram_gib,
}
(OUT/'environment.json').write_text(json.dumps(env,indent=2),encoding='utf-8')
print(json.dumps(env,indent=2))


## Tables

In [ ]:

table1=pd.DataFrame([
    ['Source','BOSSbase-derived grayscale JPEG QF95'],
    ['Image size','256 × 256 pixels'],
    ['Test images','2000'],
    ['Net payloads (bpp)','0.003, 0.006, 0.009, 0.012'],
    ['Strategies','raster; random; predictability; detectability; joint'],
    ['Joint allocator alpha','0.25'],
    ['Primary confirmatory payload','0.009 net bpp'],
    ['Independent detector','frozen enhanced residual CNN'],
    ['Primary statistical unit','paired source image'],
    ['Bootstrap resamples',str(primary['bootstrap_resamples'])],
    ['Confidence level',str(primary['confidence'])],
    ['Retuning after test','not permitted'],
],columns=['Item','Value'])
table1.to_csv(OUT/'table1_protocol.csv',index=False)
display(table1)

table2=rdh[[
    'strategy','target_net_bpp','n','actual_net_bpp_mean',
    'psnr_mean','psnr_median','ssim_mean','ssim_median',
    'encode_ms_median','decode_ms_median','used_blocks_mean',
    'changed_pixels_mean'
]].copy()
table2.to_csv(OUT/'table2_rdh_quality.csv',index=False)
table2.to_latex(OUT/'table2_rdh_quality.tex',index=False,float_format='%.4f')

table3=det[[
    'strategy','target_net_bpp','pairs',
    'auc','auc_ci_low','auc_ci_high',
    'tpr_at_5pct_fpr','tpr_ci_low','tpr_ci_high',
    'score_delta_mean','score_delta_median'
]].copy()
table3.to_csv(OUT/'table3_detectability.csv',index=False)
table3.to_latex(OUT/'table3_detectability.tex',index=False,float_format='%.4f')

table4=compact_metric_table(rdh,det,0.009)
table4.to_csv(OUT/'table4_primary_payload_tradeoff.csv',index=False)
table4.to_latex(OUT/'table4_primary_payload_tradeoff.tex',index=False,float_format='%.4f')

table5=timing[[
    'strategy','n','order_ms_median','block_analysis_ms_median',
    'encode_ms_median','decode_ms_median',
    'sender_compute_ms_median','roundtrip_compute_ms_median',
    'sender_compute_ms_p95'
]].copy()
table5.to_csv(OUT/'table5_computational_cost.csv',index=False)
table5.to_latex(OUT/'table5_computational_cost.tex',index=False,float_format='%.3f')

table6=side[[
    'strategy','target_net_bpp','n',
    'gross_payload_bits_mean','net_payload_bits_mean','sideinfo_bits_mean',
    'sideinfo_fraction_gross_mean','sideinfo_per_net_mean','used_blocks_mean'
]].copy()
table6.to_csv(OUT/'table6_sideinfo_overhead.csv',index=False)
table6.to_latex(OUT/'table6_sideinfo_overhead.tex',index=False,float_format='%.4f')

print('Table 2'); display(table2)
print('Table 3'); display(table3)
print('Table 4'); display(table4)
print('Table 5'); display(table5)
print('Table 6'); display(table6)


## Publication figures

In [ ]:

fig,ax=plt.subplots(figsize=(6.6,4.5))
for strategy in sorted(rdh.strategy.unique()):
    z=rdh[rdh.strategy==strategy].sort_values('target_net_bpp')
    ax.plot(z.target_net_bpp,z.psnr_mean,marker='o',label=strategy)
ax.set_xlabel('Net payload (bpp)')
ax.set_ylabel('Mean PSNR (dB)')
ax.set_title('Distortion versus net payload')
ax.grid(True,alpha=.2); ax.legend()
fig.tight_layout()
fig.savefig(OUT/'figure1_psnr_vs_payload.png',dpi=300)
fig.savefig(OUT/'figure1_psnr_vs_payload.svg')
plt.show()

fig,ax=plt.subplots(figsize=(6.6,4.5))
for strategy in sorted(det.strategy.unique()):
    z=det[det.strategy==strategy].sort_values('target_net_bpp')
    ax.plot(z.target_net_bpp,z.auc,marker='o',label=strategy)
ax.axhline(.5,linewidth=1)
ax.set_xlabel('Net payload (bpp)')
ax.set_ylabel('Enhanced-CNN ROC-AUC')
ax.set_title('Independent detectability versus net payload')
ax.grid(True,alpha=.2); ax.legend()
fig.tight_layout()
fig.savefig(OUT/'figure2_auc_vs_payload.png',dpi=300)
fig.savefig(OUT/'figure2_auc_vs_payload.svg')
plt.show()

fig,ax=plt.subplots(figsize=(6.6,4.5))
for strategy in sorted(det.strategy.unique()):
    z=det[det.strategy==strategy].sort_values('target_net_bpp')
    ax.plot(z.target_net_bpp,z.tpr_at_5pct_fpr,marker='o',label=strategy)
ax.axhline(.05,linewidth=1)
ax.set_xlabel('Net payload (bpp)')
ax.set_ylabel('TPR at 5% FPR')
ax.set_title('Detection at fixed false-positive rate')
ax.grid(True,alpha=.2); ax.legend()
fig.tight_layout()
fig.savefig(OUT/'figure3_tpr5_vs_payload.png',dpi=300)
fig.savefig(OUT/'figure3_tpr5_vs_payload.svg')
plt.show()


In [ ]:

trade=table4.copy()
fig,ax=plt.subplots(figsize=(6.6,4.8))
ax.scatter(trade.psnr_mean,trade.auc,s=55)
for _,r in trade.iterrows():
    ax.annotate(r.strategy,(r.psnr_mean,r.auc),xytext=(5,5),textcoords='offset points')
ax.set_xlabel('Mean PSNR at 0.009 net bpp (dB)')
ax.set_ylabel('Enhanced-CNN ROC-AUC')
ax.set_title('Distortion–detectability trade-off at 0.009 net bpp')
ax.grid(True,alpha=.2)
fig.tight_layout()
fig.savefig(OUT/'figure4_psnr_auc_tradeoff_0009.png',dpi=300)
fig.savefig(OUT/'figure4_psnr_auc_tradeoff_0009.svg')
plt.show()

z=timing.sort_values('sender_compute_ms_median')
fig,ax=plt.subplots(figsize=(6.8,4.5))
ax.bar(z.strategy,z.sender_compute_ms_median)
ax.set_ylabel('Median sender compute time (ms)')
ax.set_title('End-to-end sender computation at 0.009 net bpp')
ax.tick_params(axis='x',rotation=25)
ax.grid(True,axis='y',alpha=.2)
fig.tight_layout()
fig.savefig(OUT/'figure5_sender_compute_time.png',dpi=300)
fig.savefig(OUT/'figure5_sender_compute_time.svg')
plt.show()

j=side[side.strategy=='joint'].sort_values('target_net_bpp')
fig,ax=plt.subplots(figsize=(6.6,4.5))
ax.plot(j.target_net_bpp,100*j.sideinfo_fraction_gross_mean,marker='o')
ax.set_xlabel('Net payload (bpp)')
ax.set_ylabel('Side information / gross payload (%)')
ax.set_title('Joint allocator side-information overhead')
ax.grid(True,alpha=.2)
fig.tight_layout()
fig.savefig(OUT/'figure6_joint_sideinfo_fraction.png',dpi=300)
fig.savefig(OUT/'figure6_joint_sideinfo_fraction.svg')
plt.show()


## Manuscript-ready synthesis

In [ ]:

bpp=0.009
jdet=det[(det.strategy=='joint') & np.isclose(det.target_net_bpp,bpp)].iloc[0]
pdet=det[(det.strategy=='predictability') & np.isclose(det.target_net_bpp,bpp)].iloc[0]
jrdh=rdh[(rdh.strategy=='joint') & np.isclose(rdh.target_net_bpp,bpp)].iloc[0]
prdh=rdh[(rdh.strategy=='predictability') & np.isclose(rdh.target_net_bpp,bpp)].iloc[0]
drdh=rdh[(rdh.strategy=='detectability') & np.isclose(rdh.target_net_bpp,bpp)].iloc[0]
jt=timing[timing.strategy=='joint'].iloc[0]
pt=timing[timing.strategy=='predictability'].iloc[0]
js=side[(side.strategy=='joint') & np.isclose(side.target_net_bpp,bpp)].iloc[0]

results_text = (
    "RESULTS — frozen held-out test experiment\n\n"
    f"Across 40,000 frozen test cases, {resource['feasible_frozen_test_cases']} cases were feasible "
    "and all feasible cases achieved exact image and exact message recovery. "
    f"The primary confirmatory comparison used {primary['pairs']} common-feasible test images at {bpp:.3f} net bpp.\n\n"
    f"At {bpp:.3f} net bpp, the joint allocator (alpha={primary['alpha']:.2f}) achieved a mean PSNR of "
    f"{jrdh.psnr_mean:.2f} dB, compared with {prdh.psnr_mean:.2f} dB for predictability-only and "
    f"{drdh.psnr_mean:.2f} dB for detectability-only.\n\n"
    f"Independent enhanced-CNN detectability was lower for the joint allocator than for predictability-only: "
    f"ROC-AUC {jdet.auc:.3f} versus {pdet.auc:.3f}, and TPR at 5% FPR "
    f"{jdet.tpr_at_5pct_fpr:.3f} versus {pdet.tpr_at_5pct_fpr:.3f}. "
    f"The pre-specified paired CNN score-change difference (joint minus predictability-only) was "
    f"{primary['primary_difference']:.4f} with a 95% bootstrap CI from "
    f"{primary['primary_ci_low']:.4f} to {primary['primary_ci_high']:.4f}.\n\n"
    f"This detectability reduction incurred computational cost. Median end-to-end sender computation at "
    f"{bpp:.3f} net bpp was {jt.sender_compute_ms_median:.1f} ms for joint versus "
    f"{pt.sender_compute_ms_median:.1f} ms for predictability-only. "
    f"The joint side-information overhead averaged {100*js.sideinfo_fraction_gross_mean:.1f}% of gross payload, "
    f"corresponding to {js.sideinfo_per_net_mean:.3f} side-information bits per net payload bit.\n\n"
    "The results support a distortion–detectability–complexity trade-off rather than an absolute optimum: "
    "predictability-only minimized distortion, detectability-only minimized independent detector response, "
    "and the frozen joint allocator occupied an intermediate operating point while preserving exact reversibility."
)

limitations_text = (
    "LIMITATIONS\n\n"
    "1. The experimental source is a BOSSbase-derived grayscale JPEG QF95 collection at 256×256 pixels. "
    "Exact reversibility is demonstrated with respect to the decoded pixel array used by the RDH system, "
    "not the original pre-JPEG source file.\n"
    "2. Side information is fully accounted for in net payload but is not yet physically embedded in a self-contained bitstream.\n"
    "3. Independent confirmatory steganalysis uses one frozen enhanced residual CNN. The SRM-derived teacher used to construct local detectability risk is not independent evidence.\n"
    "4. Detectability is reduced, not eliminated; at the highest tested payload the joint allocator remains detectable above chance.\n"
    "5. Detectability-aware ordering is computationally expensive in the current implementation because local risk estimation dominates sender computation.\n"
    "6. Reported RSS boundary deltas are not peak-memory measurements and should not be presented as peak memory consumption.\n"
    "7. The test split was held out from allocator and detector selection; no post-test retuning was permitted."
)

(OUT/'results_manuscript_ready.txt').write_text(results_text,encoding='utf-8')
(OUT/'limitations_manuscript_ready.txt').write_text(limitations_text,encoding='utf-8')
print(results_text)
print()
print(limitations_text)


In [ ]:

publication_summary={
    'status':'PUBLICATION_SYNTHESIS_COMPLETE',
    'dataset':'BOSSbase-derived grayscale JPEG QF95, 256x256',
    'test_images':2000,
    'alpha':float(primary['alpha']),
    'payloads':[0.003,0.006,0.009,0.012],
    'primary_payload_bpp':float(primary['primary_payload_bpp']),
    'primary_pairs':int(primary['pairs']),
    'primary_difference':float(primary['primary_difference']),
    'primary_ci':[float(primary['primary_ci_low']),float(primary['primary_ci_high'])],
    'joint_auc_0009':float(jdet.auc),
    'predictability_auc_0009':float(pdet.auc),
    'joint_tpr5_0009':float(jdet.tpr_at_5pct_fpr),
    'predictability_tpr5_0009':float(pdet.tpr_at_5pct_fpr),
    'joint_psnr_0009':float(jrdh.psnr_mean),
    'predictability_psnr_0009':float(prdh.psnr_mean),
    'detectability_psnr_0009':float(drdh.psnr_mean),
    'joint_sender_ms_median_0009':float(jt.sender_compute_ms_median),
    'predictability_sender_ms_median_0009':float(pt.sender_compute_ms_median),
    'joint_sideinfo_fraction_gross_0009':float(js.sideinfo_fraction_gross_mean),
    'exact_recovery_feasible_cases':int(resource['feasible_frozen_test_cases']),
    'metadata_physically_embedded':bool(resource['metadata_physically_embedded']),
    'no_retuning_permitted':True,
}
(OUT/'publication_summary.json').write_text(json.dumps(publication_summary,indent=2),encoding='utf-8')
print(json.dumps(publication_summary,indent=2))



## Interpretation boundary

The paper should emphasize **reduction in detectability**, not “undetectability” or “security.” The strongest result is the held-out, paired, independent-detector comparison at the pre-specified 0.009 net bpp endpoint.

The central scientific message is:

> Highly predictable regions are attractive from a distortion perspective but are not necessarily the least detectable. A frozen joint allocation rule that explicitly incorporates local detectability risk can reduce independent steganalysis response while preserving exact reversibility, at the cost of additional computation and some distortion relative to predictability-only allocation.

No further algorithmic changes should be made after this point for the reported experiment.
